In [1]:
%load_ext watermark


In [2]:
import ast
import os
import re
import tarfile
import urllib

import pandas as pd


In [3]:
%watermark -diwmuv -iv


Last updated: 2025-06-17T05:23:14.291271+00:00

Python implementation: CPython
Python version       : 3.10.12
IPython version      : 7.31.1

Compiler    : GCC 11.4.0
OS          : Linux
Release     : 6.8.0-1029-azure
Machine     : x86_64
Processor   : x86_64
CPU cores   : 4
Architecture: 64bit

re     : 2.2.1
pandas : 2.2.3
tarfile: 0.9.0

Watermark: 2.4.3



In [4]:
pd.options.display.float_format = "{:,.0f}".format


In [5]:
summary_start = re.compile(r"^Simulation summary:", re.MULTILINE)
param_dict = re.compile(r"\{(?:[^{}]|\n)*\}", re.DOTALL)


def parse_file(path: str) -> dict:
    text = open(path, "r", encoding="utf-8", errors="ignore").read()
    # a) find summary block
    m = summary_start.search(text)
    if not m:
        return None
    # read lines after “Simulation summary:” until a blank line
    lines = text[m.end() :].splitlines()
    summary_lines = []
    for ln in lines[1:]:
        if not ln.strip():
            break
        summary_lines.append(ln)

    # b) find param dict
    d_match = param_dict.search(text)
    if not d_match:
        return None

    # parse metrics
    metrics = {}
    for ln in summary_lines:
        parts = ln.strip().split()
        # first token is the number (with commas), rest is the metric name
        val = int(parts[0].replace(",", ""))
        key = "_".join(parts[1:])
        metrics[key] = val

    # parse params dict
    params = ast.literal_eval(d_match.group(0))

    # combine
    return {**params, **metrics}


In [6]:
def make_df(slug: str) -> pd.DataFrame:
    # 1. Download the tar.gz
    url = f"https://osf.io/{slug}/download"
    archive_path = f"{slug}.tar.gz"
    urllib.request.urlretrieve(url, archive_path)

    # 2. Extract into ./data/
    extract_dir = slug
    os.makedirs(extract_dir, exist_ok=True)
    with tarfile.open(archive_path, mode="r:gz") as tar:
        tar.extractall(extract_dir)

    # 3. Walk data folder, parse all files
    records = []
    for root, _, files in os.walk(slug):
        for fn in files:
            full = os.path.join(root, fn)
            rec = parse_file(full)
            if rec:
                records.append(rec)

    # 4. Build DataFrame
    return pd.DataFrame(records)


## Vanilla


In [7]:
slug = "tmg6b"
df = make_df(slug)
df.to_csv(f"{slug}.csv", index=False)


In [8]:
df.loc[:, df.columns.str.startswith("cumulative_")].mean(
    numeric_only=True
).to_frame("mean")


,mean
cumulative_infections,"709,268"
cumulative_reinfections,"514,031"
cumulative_infectious,"705,985"
cumulative_symptomatic_cases,"380,888"
cumulative_severe_cases,"22,987"
cumulative_critical_cases,"6,962"
cumulative_recoveries,"696,970"
cumulative_deaths,"2,498"
cumulative_tests,0
cumulative_diagnoses,0


In [9]:
df.loc[:, df.columns.str.startswith("cumulative_")].std(
    numeric_only=True
).to_frame("std")


,std
cumulative_infections,"9,553"
cumulative_reinfections,"9,352"
cumulative_infectious,"9,498"
cumulative_symptomatic_cases,"4,503"
cumulative_severe_cases,245
cumulative_critical_cases,103
cumulative_recoveries,"9,358"
cumulative_deaths,59
cumulative_tests,0
cumulative_diagnoses,0


## Vanilla --- big


In [10]:
slug = "czt4b"
df = make_df(slug)
df.to_csv(f"{slug}.csv", index=False)


In [11]:
df.loc[:, df.columns.str.startswith("cumulative_")].mean(
    numeric_only=True
).to_frame("mean")


,mean
cumulative_infections,"4,212,074"
cumulative_reinfections,"3,041,007"
cumulative_infectious,"4,192,348"
cumulative_symptomatic_cases,"2,263,508"
cumulative_severe_cases,"136,584"
cumulative_critical_cases,"41,519"
cumulative_recoveries,"4,138,274"
cumulative_deaths,"14,832"
cumulative_tests,0
cumulative_diagnoses,0


In [12]:
df.loc[:, df.columns.str.startswith("cumulative_")].std(
    numeric_only=True
).to_frame("std")


,std
cumulative_infections,"15,273"
cumulative_reinfections,"14,867"
cumulative_infectious,"15,245"
cumulative_symptomatic_cases,"7,484"
cumulative_severe_cases,400
cumulative_critical_cases,255
cumulative_recoveries,"14,919"
cumulative_deaths,104
cumulative_tests,0
cumulative_diagnoses,0


## UK


In [13]:
slug = "ej5bz"
df = make_df(slug)
df.to_csv(f"{slug}.csv", index=False)


In [14]:
df.loc[:, df.columns.str.startswith("cumulative_")].mean(
    numeric_only=True
).to_frame(
    "mean"
) * 200_000 / 55.98e6  # UK population size


,mean
cumulative_infections,"18,902"
cumulative_reinfections,"1,526"
cumulative_infectious,"18,903"
cumulative_symptomatic_cases,"12,900"
cumulative_severe_cases,"1,210"
cumulative_critical_cases,400
cumulative_recoveries,"18,746"
cumulative_deaths,168
cumulative_tests,"591,395"
cumulative_diagnoses,"3,832"


In [15]:
df.loc[:, df.columns.str.startswith("cumulative_")].std(
    numeric_only=True
).to_frame(
    "std"
) * 200_000 / 55.98e6  # UK population size


,std
cumulative_infections,"4,857"
cumulative_reinfections,572
cumulative_infectious,"4,859"
cumulative_symptomatic_cases,"3,282"
cumulative_severe_cases,305
cumulative_critical_cases,104
cumulative_recoveries,"4,817"
cumulative_deaths,44
cumulative_tests,"2,453"
cumulative_diagnoses,"1,513"


## UK --- big


In [16]:
slug = "nm6wz"
df = make_df(slug)
df.to_csv(f"{slug}.csv", index=False)


In [17]:
df.loc[:, df.columns.str.startswith("cumulative_")].mean(
    numeric_only=True
).to_frame(
    "mean"
) * 1_200_000 / 55.98e6  # UK population size


,mean
cumulative_infections,"93,412"
cumulative_reinfections,"6,897"
cumulative_infectious,"93,420"
cumulative_symptomatic_cases,"63,780"
cumulative_severe_cases,"5,977"
cumulative_critical_cases,"1,987"
cumulative_recoveries,"92,575"
cumulative_deaths,845
cumulative_tests,"3,516,652"
cumulative_diagnoses,"19,426"


In [18]:
df.loc[:, df.columns.str.startswith("cumulative_")].std(
    numeric_only=True
).to_frame(
    "std"
) * 1_200_000 / 55.98e6  # UK population size


,std
cumulative_infections,"13,418"
cumulative_reinfections,"1,384"
cumulative_infectious,"13,416"
cumulative_symptomatic_cases,"9,064"
cumulative_severe_cases,859
cumulative_critical_cases,299
cumulative_recoveries,"13,301"
cumulative_deaths,127
cumulative_tests,"6,642"
cumulative_diagnoses,"4,009"


## multistrain


In [19]:
slug = "qdwb7"
df = make_df(slug)
df.to_csv(f"{slug}.csv", index=False)


In [20]:
df.loc[:, df.columns.str.startswith("cumulative_")].mean(
    numeric_only=True
).to_frame("mean")


,mean
cumulative_infections,"1,212,595"
cumulative_reinfections,"1,012,835"
cumulative_infectious,"1,204,795"
cumulative_symptomatic_cases,"609,641"
cumulative_severe_cases,"20,162"
cumulative_critical_cases,"6,081"
cumulative_recoveries,"1,187,836"
cumulative_deaths,"2,192"
cumulative_tests,0
cumulative_diagnoses,0


In [21]:
df.loc[:, df.columns.str.startswith("cumulative_")].std(
    numeric_only=True
).to_frame("std")


,std
cumulative_infections,"5,645"
cumulative_reinfections,"5,637"
cumulative_infectious,"5,665"
cumulative_symptomatic_cases,"2,716"
cumulative_severe_cases,251
cumulative_critical_cases,104
cumulative_recoveries,"5,653"
cumulative_deaths,45
cumulative_tests,0
cumulative_diagnoses,0
